# PN10B child phase inside the pre-ridge factor sphere

## tl;dr

The registered child decomposition closed exactly but produced a **NULL** prime-ranking result. On the fresh
interval, ARA full scored `0.652923909` bits per survivor versus `0.652816910` for the parent-only forecast, with
ROC AUC `0.500307`. The 95% paired interval included zero. An independent implementation passed 79/79 checks.

## Context & Methods

PN10B used only the nine largest divisor gates already tested at parent cutoff `c=0.90`. For each gate,

\[
A_j=2(n\bmod q_j)/q_j,\quad B_j=2-A_j,\quad s_j=A_j-1,\quad h_j=s_js_{j+1}.
\]

The primary model used nine ordered child orientations and eight adjacent couplings. It was frozen before opening
the fresh target `[4,000,000,000,4,001,000,000)`.

### Key Assumptions

- `A` and `B` are two directions of one child axis, not independent features.
- No gate above `n^0.45` may be inspected.
- Positive performance would mean useful organisation of existing information, not new Shannon information.


In [1]:
from pathlib import Path
import csv, hashlib, json

HERE = Path(r'F:\SystemFormulaFolder\GIT\ARA-GIT\analysis\primes')
freeze = json.loads((HERE / 'PN10B_FREEZE_MANIFEST.json').read_text(encoding='utf-8'))
result = json.loads((HERE / 'PN10B_CHILD_PHASE_RESULTS.json').read_text(encoding='utf-8'))
validation = json.loads((HERE / 'PN10B_CHILD_PHASE_VALIDATION.json').read_text(encoding='utf-8'))
sha = lambda path: hashlib.sha256(path.read_bytes()).hexdigest().upper()
print('Protocol hash matches:', sha(HERE / freeze['protocol_file']) == freeze['protocol_sha256'])
print('Source hash matches:  ', sha(HERE / freeze['source_file']) == freeze['source_sha256'])
print('Fresh target:', freeze['intervals']['F'])
print('Protected:', result['protected_material'])


Protocol hash matches: True
Source hash matches:   True
Fresh target: [4000000000, 4001000000]
Protected: {'p31_primorial_wheel_constructed': False, 'r12_opened': False}


In [2]:
print('## Data')
for name, interval in result['intervals'].items():
    print(name, 'survivors=', interval['survivor_count'], 'primes=', interval['prime_count'],
          'composites=', interval['composite_count'], 'purity=', round(interval['prime_prevalence'], 9))
    print('  guards:', interval['guards'])


## Data
D survivors= 84117 primes= 70435 composites= 13682 purity= 0.837345602
  guards: {'max_abs_a_plus_b_minus_2': 0.0, 'max_gate_minus_threshold': -0.00011349374051405903, 'zero_remainders': 0, 'all_gates_already_tested': True}
E survivors= 56152 primes= 46903 composites= 9249 purity= 0.835286366
  guards: {'max_abs_a_plus_b_minus_2': 0.0, 'max_gate_minus_threshold': -1.0960546205751598e-05, 'zero_remainders': 0, 'all_gates_already_tested': True}
F survivors= 54275 primes= 45166 composites= 9109 purity= 0.832169507
  guards: {'max_abs_a_plus_b_minus_2': 0.0, 'max_gate_minus_threshold': -2.7020967536373064e-05, 'zero_remainders': 0, 'all_gates_already_tested': True}


In [3]:
print('## Results')
metrics = [row for row in result['metrics'] if row['stage'] == 'pooled_D_E_to_fresh_F']
print('model | log loss bits | Brier | AUC | top-decile lift')
for row in metrics:
    print(f"{row['model']:<23} {row['log_loss_bits']:.9f} {row['brier']:.9f} {row['auc']:.6f} {row['top_decile_lift']:.6f}")


## Results
model | log loss bits | Brier | AUC | top-decile lift
parent_empirical        0.652816910 0.139682356 0.500000 0.994683
buchstab_parent         0.652720245 0.139663906 0.500000 0.994683
ara_compact             0.652846784 0.139687805 0.501193 0.995569
raw_compact             0.652902634 0.139698370 0.497254 0.999111
ara_full                0.652923909 0.139702407 0.500307 1.012394
raw_full                0.652923873 0.139702352 0.496451 0.999775
ara_order_scrambled     0.652841097 0.139686710 0.502502 1.005088


In [4]:
print('Fresh paired comparisons:')
for name, row in result['fresh_comparisons'].items():
    print(f"{name:<42} gain={row['gain_bits_per_event']:+.9f} "
          f"CI=[{row['ci95_low']:+.9f},{row['ci95_high']:+.9f}] blocks+={row['positive_blocks']}/100")
print('Criteria:', result['criteria'])
print('Verdict:', result['verdict'])


Fresh paired comparisons:
ara_full_vs_parent_empirical               gain=-0.000106999 CI=[-0.000241111,+0.000034314] blocks+=40/100
ara_full_vs_raw_full                       gain=-0.000000036 CI=[-0.000074170,+0.000070622] blocks+=55/100
ara_full_vs_ara_order_scrambled            gain=-0.000082811 CI=[-0.000208106,+0.000032616] blocks+=44/100
ara_compact_vs_parent_empirical            gain=-0.000029874 CI=[-0.000125758,+0.000057329] blocks+=50/100
ara_compact_vs_raw_compact                 gain=+0.000055850 CI=[+0.000011362,+0.000103680] blocks+=59/100
raw_full_vs_parent_empirical               gain=-0.000106963 CI=[-0.000220148,+0.000010932] blocks+=39/100
Criteria: {'P1': True, 'P2': False, 'P3': False, 'P4': False, 'P5': False, 'P6': False}
Verdict: NULL


In [5]:
print('## Validation')
print('Independent checks:', validation['checks_passed'], '/', validation['checks_total'])
print('All passed:', validation['all_passed'])
print('Maximum fitted gradient:', validation['max_fitted_gradient'])
print('Maximum metric disagreement:', validation['max_metric_error'])
print('Static figure:', HERE / 'PN10B_CHILD_PHASE_FIGURE.png')


## Validation
Independent checks: 79 / 79
All passed: True
Maximum fitted gradient: 1.1554253408435247e-17
Maximum metric disagreement: 0.0
Static figure: F:\SystemFormulaFolder\GIT\ARA-GIT\analysis\primes\PN10B_CHILD_PHASE_FIGURE.png


## Takeaways

1. The child A/B axes are mathematically valid and leak-free at the registered gate budget.
2. Their ordered coupling did not rank fresh primes above remaining composites; ARA full was effectively chance.
3. Buchstab's constant parent probability calibrated best but, as a constant, also did not rank individuals.
4. The immediate hypothesis is closed as a clean null. A different child identity would require a new registration.
